# Phase 5 — Evaluation (part 1)
No DeepSeek calls. In-library title benchmark and optional paired demo; not independent validation. Challenge manifest needs SAP-body verification.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
import tfl_agent
from tfl_agent.agent.sap_scope_pilot import run_scope_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)
from tfl_agent.agent.phase4_pipeline import run_phase4
from tfl_agent.agent.phase5_evaluation import run_phase5

## Load previous Phase 4 result from Drive
The two demo labels are provisional expectations, not independently adjudicated gold labels.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, zipfile, hashlib
from datetime import datetime, timezone
SAP_PATH = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo/SAP_001_00.pdf')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/Phase5') / RUN_ID
INPUT = Path('/content/Phase4_Input') / RUN_ID
ZIP_NAME = '' #@param {type:"string"}
available = sorted((SAP_PATH.parent/'Phase4_Outputs').glob('*.zip'))
labels=[]
if available:
    zpath=SAP_PATH.parent/'Phase4_Outputs'/ZIP_NAME if ZIP_NAME else available[-1]
    INPUT.mkdir(parents=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.infolist():
            if not (INPUT/m.filename).resolve().is_relative_to(INPUT.resolve()):
                raise ValueError('Unsafe ZIP path')
        z.extractall(INPUT)
    h=hashlib.sha256(SAP_PATH.read_bytes()).hexdigest()
    assert json.loads((INPUT/'Summary.json').read_text())['sap_sha256']==h
    labels=[
      {'study_id':'demo_SAP_001_00','output_id':'14.1.3.1','title':'Demographic and Baseline Characteristics – Safety Analysis Set','output_type':'table','expected':'TBL_VARIABLE_SUMMARY','sap_sha256':h},
      {'study_id':'demo_SAP_001_00','output_id':'14.2.3.1','title':'Summary of Progression Free Survival – ITT Analysis Set','output_type':'table','expected':'TBL_TIME_TO_EVENT','sap_sha256':h}
    ]
else:
    print('No Phase 4 ZIP; title benchmark only.')

## Run title benchmark and paired demonstration

In [ ]:
import pandas as pd
from IPython.display import display
summary, rows, challenge = run_phase5(ROOT, OUT, paired_dir=INPUT if labels else None, paired_labels=labels)
print(json.dumps(summary,indent=2))
display(pd.DataFrame(summary['title_benchmark']).T)
display(pd.DataFrame(summary['title_challenge_subset']).T)
display(pd.DataFrame(challenge)[['study_id','output_id','title','expected','predicted','outcome','available_pdf_paths']])
if summary['paired_demo']:
    display(pd.DataFrame(summary['paired_demo']).T)

## Regression tests

In [ ]:
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_phase5_evaluation.py','tests/test_sap_aware_retrieval_phase4.py','tests/test_generic_library_phase1.py'],cwd=ROOT)

## Export results
Run export even if a regression test fails after evaluation.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH = Path(shutil.make_archive(str(OUT), 'zip', OUT))
destination = SAP_PATH.parent / 'Phase5_Outputs'
destination.mkdir(parents=True, exist_ok=True)
saved = Path(shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name))
print('Saved to Drive:', saved)
files.download(str(ZIP_PATH))